<a href="https://colab.research.google.com/github/Qashaisya/mcta4364-yolov10-conveyor-sorting/blob/main/notebooks/02_yolov10_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!pip install -q ultralytics roboflow

Thu Nov 26 17:54:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os, yaml
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("custom-3w4p0").project("conveyor-belt-object-detection-ehaa8")
version = project.version(9)
dataset = version.download("yolov8")

DATA_DIR = dataset.location
print("Dataset downloaded to:", DATA_DIR)

with open(f"{DATA_DIR}/data.yaml", "r") as f:
    cfg = yaml.safe_load(f)
print("Classes:", cfg.get("names"))
print("Splits found:", [s for s in ["train", "valid", "test"] if os.path.exists(f"{DATA_DIR}/{s}")])

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to conveyor-belt-object-detection-9 in yolov8:: 100%|██████████| 5712/5712 [00:00<00:00, 5907.01it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Dataset downloaded to: /content/conveyor-belt-object-detection-9
Classes: ['anomaly', 'glass', 'metal', 'plastic']
Splits found: ['train', 'valid', 'test']


In [3]:
import os
from ultralytics import YOLO

model = YOLO("yolov10n.pt")

# Train reproducible baseline (~5-10 mins on T4 GPU)
model.train(
    data=f"{DATA_DIR}/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    seed=42,
    plots=True,
    project="/content/runs/detect",
    name="yolov10n_baseline"
)

# Use test split if available, otherwise val
eval_split = "test" if os.path.exists(f"{DATA_DIR}/test/images") else "val"
print(f"\nEvaluating on '{eval_split}' split...")

metrics = model.val(
    data=f"{DATA_DIR}/data.yaml",
    split=eval_split,
    imgsz=640,
    plots=True,
    project="/content/runs/detect",
    name="yolov10n_eval"
)

p, r = metrics.box.mp, metrics.box.mr
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

print("\n=== COPY THESE METRICS FOR SECTION 4 ===")
print(f"Eval Split: {eval_split}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"F1-Score:   {f1:.4f}")
print(f"mAP@50:     {metrics.box.map50:.4f}")
print(f"mAP@50-95:  {metrics.box.map:.4f}")
print(f"Inference:  {metrics.speed['inference']:.2f} ms/img")

Ultralytics 8.4.219 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/conveyor-belt-object-detection-9/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov10n_baseline, nbs=64, nms=None,

In [4]:
import shutil
from google.colab import files

shutil.make_archive("/content/baseline_results", "zip", "/content/runs/detect")
files.download("/content/baseline_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>